# Integrating a PSD Gives You $\sigma$ — and Throws Away Everything Else

There is a common piece of folklore that goes roughly:

> *"You can't just integrate the PSD to get the time-domain standard deviation — that only works for white
> Gaussian noise."*

The intuition behind this warning is sound, but the statement itself is false, and it is worth separating the two.

**What is actually true:**

1. **Integrating the PSD always gives the variance.** For *any* wide-sense-stationary (WSS) process,
   $$\sigma^2 = \int_0^{\infty} S_x(f)\,df$$
   This is the Wiener–Khinchin theorem evaluated at zero lag. It does not require whiteness, and it does not
   require Gaussianity. Colored noise, flicker noise, a narrow bandpass process — the identity holds for all of them.

2. **But the map $S_x(f) \mapsto \sigma$ is many-to-one, and therefore lossy.** Infinitely many different spectra
   integrate to the same $\sigma$. Collapsing a PSD to a single number discards *how the power is distributed
   across frequency* — and that distribution is what governs the process's dynamics in time.

**A separate confusion worth clearing up:** "Gaussian" and "white" are independent properties.

| | meaning | describes |
|---|---|---|
| **Gaussian** | the amplitude at each instant is normally distributed | the *marginal distribution* |
| **White** | samples are mutually uncorrelated; PSD is flat | the *correlation structure* / spectrum |

A Gaussian process is **not** necessarily white. Passing white Gaussian noise through any linear filter
leaves it *exactly* Gaussian (linear combinations of jointly Gaussian variables are Gaussian) while making the
PSD arbitrarily colored. So "we assume the noise is Gaussian" tells you nothing at all about the PSD shape.

This notebook demonstrates both points numerically: we construct four Gaussian processes with **identical
$\sigma = 1$** but radically different spectra, confirm that each one's PSD integrates back to $\sigma^2$, and
then show how differently they behave in time.

## Setup

We synthesize each process by shaping the spectrum of white noise: draw uniform random phase, impose the
desired magnitude $\sqrt{S(f)}$, and inverse-FFT. The result is a genuine Gaussian process (a sum of many
independent random-phase sinusoids — the CLT does the rest) with the PSD we asked for.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

rng = np.random.default_rng(2718)

fs = 1e6                      # 1 MS/s
N  = 1 << 20                  # ~1.05e6 samples
f  = np.fft.rfftfreq(N, 1/fs)
f[0] = f[1]                   # avoid div-by-zero at DC for the 1/f shapes

SIGMA = 1.0                   # every process is forced to this exact std dev

def synth(shape, sigma=SIGMA):
    """Gaussian process with one-sided PSD proportional to shape(f), renormalized to `sigma`."""
    S  = shape(f)
    ph = rng.uniform(0, 2*np.pi, f.size)
    X  = np.sqrt(S) * np.exp(1j*ph)
    X[0] = 0                                  # zero mean
    x  = np.fft.irfft(X, n=N)
    return x / x.std() * sigma                # <- identical variance by construction

shapes = {
    "white":               lambda f: np.ones_like(f),
    "1/f (flicker)":       lambda f: 1/f,
    "1/f² (random walk)":  lambda f: 1/f**2,
    "bandpass (40–60 kHz)": lambda f: ((f > 4e4) & (f < 6e4)).astype(float) + 1e-12,
}

procs = {name: synth(sh) for name, sh in shapes.items()}
print(f"{'process':24s} {'mean':>9s} {'std':>8s}")
for name, x in procs.items():
    print(f"{name:24s} {x.mean():9.2e} {x.std():8.4f}")

All four have the same standard deviation. If $\sigma$ were a sufficient description of a noise process,
these would now be interchangeable. They are not.

## 1. Parseval holds for every one of them

Integrating each PSD recovers $\sigma^2 = 1$ regardless of spectral shape — the identity is not special to
white noise.

In [ ]:
print(f"{'process':24s} {'∫S(f)df':>10s} {'σ² (direct)':>13s} {'error':>10s}")
for name, x in procs.items():
    fp, P = signal.periodogram(x, fs=fs)      # full resolution -> see note below
    integral = np.trapezoid(P, fp)
    var      = x.var()
    print(f"{name:24s} {integral:10.4f} {var:13.4f} {abs(integral-var):10.2e}")

> **A numerical caveat that matters in practice.** We used `periodogram` (full frequency resolution) rather than
> `welch`. Welch's method splits the record into segments, so its lowest resolvable frequency is $f_s/\texttt{nperseg}$
> — any power below that bin is simply not counted. For steep spectra like $1/f^2$, where most of the power lives at
> the very lowest frequencies, this silently loses most of the variance. This is a *measurement* artifact, not a
> failure of Parseval, and it is exactly the trap that makes people distrust the PSD→$\sigma$ conversion.

In [ ]:
print("∫PSD df using Welch at various segment lengths (true value = 1.0):\n")
segs = [8192, 65536, 262144]
print(f"{'process':24s}" + "".join(f"{'nperseg='+str(n):>16s}" for n in segs) + f"{'periodogram':>14s}")
for name, x in procs.items():
    row = f"{name:24s}"
    for nps in segs:
        fw, P = signal.welch(x, fs=fs, nperseg=nps)
        row += f"{np.trapezoid(P, fw):16.4f}"
    fp, P = signal.periodogram(x, fs=fs)
    row += f"{np.trapezoid(P, fp):14.4f}"
    print(row)
print("\nThe 1/f² row collapses toward zero as resolution coarsens: its power is below the lowest Welch bin.")

## 2. Identical $\sigma$, completely different processes

Here is the actual point. Same variance, same Gaussian marginal — but the time-domain character is unrecognizably
different, and that difference is precisely the information the integral discarded.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(13, 9),
                         gridspec_kw={"width_ratios": [2, 1]})

# Left: each process over a window matched to its own correlation time, so the
# character of the wander is visible. (In a 4k-sample window the 1/f^2 process
# would look like a flat line -- it drifts ~50k samples at a time.)
windows = {"white": 4000, "1/f (flicker)": 20000,
           "1/f² (random walk)": N, "bandpass (40–60 kHz)": 4000}

# Right: the marginal is pooled over many independent realizations. A single
# strongly-correlated record contains too few independent excursions for its
# histogram to converge, even though the process is Gaussian by construction.
N_REAL = 40

for ax_row, (name, x) in zip(axes, procs.items()):
    ax_t, ax_h = ax_row
    n_show = windows[name]
    t = np.arange(n_show) / fs * 1e3      # ms

    ax_t.plot(t, x[:n_show], lw=0.6)
    ax_t.axhline( SIGMA, color="k", ls="--", lw=0.8, alpha=0.6)
    ax_t.axhline(-SIGMA, color="k", ls="--", lw=0.8, alpha=0.6)
    ax_t.set_ylim(-4.5, 4.5)
    ax_t.set_ylabel(name, fontsize=9)
    ax_t.margins(x=0)
    ax_t.text(0.99, 0.04, f"{n_show/fs*1e3:.4g} ms window", transform=ax_t.transAxes,
              ha="right", va="bottom", fontsize=8, alpha=0.65)

    pooled = np.concatenate([synth(shapes[name]) for _ in range(N_REAL)])
    ax_h.hist(pooled, bins=140, density=True, alpha=0.75, orientation="horizontal")
    g = np.linspace(-4.5, 4.5, 300)
    ax_h.plot(np.exp(-g**2/2)/np.sqrt(2*np.pi), g, "r-", lw=1.2)
    ax_h.set_ylim(-4.5, 4.5)
    ax_h.set_xticks([])

axes[0][0].set_title("time domain (dashed = ±1σ, identical for all four)")
axes[0][1].set_title(f"marginal, {N_REAL} realizations pooled (red = Gaussian)")
axes[-1][0].set_xlabel("time [ms]")
plt.tight_layout()
plt.show()


Every histogram on the right matches the same Gaussian curve — all four processes are Gaussian with $\sigma=1$.
Yet on the left, the white process rattles between its bounds every sample, the $1/f^2$ process wanders slowly and
may not cross zero for the whole window, and the bandpass process is a visible 50 kHz burst. **A single $\sigma$
cannot distinguish these.**

## 3. The spectra that produced them

In [ ]:
plt.figure(figsize=(9, 5))
for name, x in procs.items():
    fw, P = signal.welch(x, fs=fs, nperseg=16384)
    plt.loglog(fw[1:], P[1:], lw=1.1, label=f"{name}  (σ={x.std():.3f})")
plt.xlabel("frequency [Hz]")
plt.ylabel("PSD  [V²/Hz]")
plt.title("Four spectra, one standard deviation")
plt.legend(fontsize=9)
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

Roughly 8 decades of spread in PSD at low frequency — all integrating to the same total power.

## 4. Why this bites in practice: the band matters, not the total

In circuit work you almost never care about total integrated noise. You care about noise **in a band**, because
a downstream system only responds to part of the spectrum. Two processes with equal $\sigma$ can contribute
wildly different amounts once a loop bandwidth or measurement window selects a region.

In [ ]:
bands = [(1e2, 1e3), (1e3, 1e4), (1e4, 1e5), (1e5, 5e5)]

print("RMS contribution per band (total σ = 1.000 for every row):\n")
hdr = f"{'process':24s}" + "".join(f"{f'{lo:.0e}-{hi:.0e}':>16s}" for lo, hi in bands)
print(hdr); print("-"*len(hdr))
for name, x in procs.items():
    fp, P = signal.periodogram(x, fs=fs)
    row = f"{name:24s}"
    for lo, hi in bands:
        m = (fp >= lo) & (fp < hi)
        row += f"{np.sqrt(np.trapezoid(P[m], fp[m])):16.4f}"
    print(row)

In [ ]:
# Correlation structure: the other thing sigma cannot tell you
print(f"{'process':24s} {'lag-1 corr':>12s} {'corr time (samples)':>22s}")
for name, x in procs.items():
    r1 = np.corrcoef(x[:-1], x[1:])[0, 1]
    xc = x[:1 << 16]
    ac = np.correlate(xc, xc, "full")[len(xc)-1:]
    ac /= ac[0]
    tau = np.argmax(ac < 1/np.e) if np.any(ac < 1/np.e) else len(ac)
    print(f"{name:24s} {r1:12.4f} {tau:22d}")

## Takeaways

1. **$\sigma^2 = \int S_x(f)\,df$ is valid for any WSS process** — Gaussian or not, white or not. The rule is
   Wiener–Khinchin, not a white-noise special case.

2. **"Gaussian" and "white" are orthogonal properties.** Gaussian describes the amplitude histogram; white
   describes the spectrum. Filtering changes the spectrum and leaves Gaussianity intact, so assuming Gaussian
   noise implies nothing about PSD shape.

3. **The conversion is lossy, and that is the real caution.** $\sigma$ is one number summarizing a whole function.
   Processes with the same $\sigma$ can differ in correlation time by orders of magnitude and put their power in
   entirely different bands.

4. **In practice, integrate over the band you care about.** For phase noise and jitter this is the whole game:
   integrating $\mathcal{L}(f)$ over different offset ranges yields different jitter figures, which is why a
   specification is meaningless without its integration limits.

5. **Watch your estimator's resolution.** Welch with short segments will under-report the variance of steep
   low-frequency-dominated spectra, because power below the first bin is never counted.

## 5. Going the other way: PSD → time-domain vector → PSD

Everything above shows what $\sigma$ *loses*. The constructive counterpart is that the PSD itself loses nothing:
given $S_x(f)$ you can synthesize a time-domain realization, and FFT it back to recover the spectrum you started
from. Here we do that round trip on a realistic composite spectrum — flicker plus a white floor:

$$S_x(f) = \frac{10^{-12}}{f} + 10^{-16} \quad \mathrm{V^2/Hz}$$

The recipe:

1. Set each bin's **magnitude** from the target PSD: $|X_k| = \sqrt{S(f_k)\,\Delta f}\cdot\frac{N}{\sqrt 2}$
2. Give each bin an **independent uniform random phase** — this is the information $S_x(f)$ does not contain
3. Force the spectrum to be **Hermitian** so the inverse transform is real (`irfft` handles this, but DC and
   Nyquist must be real: set $X_0=0$ for zero mean, and $X_{N/2}$ real)
4. Inverse-FFT

In [ ]:
rng_rt = np.random.default_rng(31415)

df = fs / N                                   # bin width
S_target = lambda f: 1e-12/f + 1e-16          # V^2/Hz, one-sided

# --- 1-3: build the spectrum ------------------------------------------------
S   = S_target(f)
mag = np.sqrt(S * df * N**2 / 2)              # numpy irfft scaling convention
ph  = rng_rt.uniform(0, 2*np.pi, f.size)      # <- the phase the PSD never knew
X   = mag * np.exp(1j*ph)
X[0]  = 0                                     # zero DC / zero mean
X[-1] = np.abs(X[-1])                         # Nyquist bin must be real

# --- 4: back to the time domain --------------------------------------------
x_rt = np.fft.irfft(X, n=N)

print(f"synthesized {N:,} samples at {fs/1e6:.0f} MS/s")
print(f"mean = {x_rt.mean():.3e}   std = {x_rt.std():.6e} V")

Here is the vector itself. The top panel is the synthesized waveform; below it, the same record at three zoom
levels, and two more realizations drawn with *different random phase from the identical PSD*.

In [ ]:
fig = plt.figure(figsize=(13, 8))
gs  = fig.add_gridspec(3, 3, hspace=0.45, wspace=0.25)

# --- full record --------------------------------------------------------
ax = fig.add_subplot(gs[0, :])
t_full = np.arange(N)/fs*1e3
ax.plot(t_full, x_rt*1e6, lw=0.35)
for s in (-1, 1):
    ax.axhline(s*x_rt.std()*1e6, color="k", ls="--", lw=0.8, alpha=0.6)
ax.set_xlim(0, t_full[-1]); ax.margins(x=0)
ax.set_title(f"synthesized from $S_x(f)$ — full record ({N:,} samples, σ = {x_rt.std()*1e6:.2f} µV, "
             f"dashed = ±1σ)", fontsize=10)
ax.set_xlabel("time [ms]"); ax.set_ylabel("x  [µV]")

# --- zooms --------------------------------------------------------------
for col, n_z in enumerate([100_000, 10_000, 1_000]):
    ax = fig.add_subplot(gs[1, col])
    ax.plot(np.arange(n_z)/fs*1e3, x_rt[:n_z]*1e6, lw=0.6)
    for s in (-1, 1):
        ax.axhline(s*x_rt.std()*1e6, color="k", ls="--", lw=0.7, alpha=0.5)
    ax.set_title(f"first {n_z:,} samples", fontsize=9)
    ax.set_xlabel("time [ms]"); ax.margins(x=0)
    if col == 0: ax.set_ylabel("x  [µV]")

# --- same PSD, different phase -----------------------------------------
for col in range(3):
    ax = fig.add_subplot(gs[2, col])
    ph2 = rng_rt.uniform(0, 2*np.pi, f.size)
    X2  = mag*np.exp(1j*ph2); X2[0] = 0; X2[-1] = np.abs(X2[-1])
    x2  = np.fft.irfft(X2, n=N)
    ax.plot(np.arange(10_000)/fs*1e3, x2[:10_000]*1e6, lw=0.6, color="C2")
    ax.set_title(f"realization #{col+2} — same $S_x(f)$, new phase (σ = {x2.std()*1e6:.2f} µV)",
                 fontsize=9)
    ax.set_xlabel("time [ms]"); ax.margins(x=0)
    if col == 0: ax.set_ylabel("x  [µV]")

plt.show()

### Does it round-trip?

Two independent checks. First, Parseval: the variance predicted from the coefficients we actually set should
equal the measured variance of the time series.

In [ ]:
var_exact = 2*np.sum(np.abs(X[1:-1])**2)/N**2 + np.abs(X[-1])**2/N**2
print(f"σ from spectral coefficients : {np.sqrt(var_exact):.9e} V")
print(f"σ measured from x_rt         : {x_rt.std():.9e} V")
print(f"relative error               : {abs(np.sqrt(var_exact)-x_rt.std())/x_rt.std():.2e}")

trap = np.sqrt(np.trapezoid(S_target(f), f))
print(f"\nσ via np.trapezoid(S, f)     : {trap:.9e} V   ({abs(trap-x_rt.std())/x_rt.std():.2%} off)")
print("The trapezoid gap is quadrature error -- 1/f varies steeply across the first")
print("few bins of a LINEAR grid -- not a failure of Parseval.")

Exact to machine precision. Now the second check: FFT the synthesized vector and compare the recovered PSD
against the target it was built from.

In [ ]:
fw_rt, P_rt = signal.welch(x_rt, fs=fs, nperseg=16384)
m = fw_rt > 0

plt.figure(figsize=(9, 5))
plt.loglog(fw_rt[m], P_rt[m], lw=0.8, alpha=0.75, label="recovered from x (Welch)")
plt.loglog(fw_rt[m], S_target(fw_rt[m]), "r--", lw=1.8, label="target $S_x(f)$")
plt.xlabel("frequency [Hz]"); plt.ylabel("PSD  [V²/Hz]")
plt.title("PSD → time domain → PSD round trip")
plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

print("  freq [Hz]        target        recovered     ratio")
for probe in [1e2, 1e3, 1e4, 1e5]:
    i = np.argmin(np.abs(fw_rt - probe))
    t_, r_ = S_target(fw_rt[i]), P_rt[i]
    print(f"  {fw_rt[i]:9.1f}   {t_:.4e}   {r_:.4e}   {r_/t_:.3f}")

The recovered spectrum lands on the target across four decades, including the $1/f$ knee. So the PSD → time-domain
map is faithful and invertible.

**The asymmetry is the whole point.** Going PSD → time domain required inventing something the PSD never carried:
the random phase of every bin. Different phase draws give completely different-looking waveforms that all share
the same PSD. And going time domain → $\sigma$ throws away even more, collapsing the entire function to one number.

$$\underbrace{x(t)}_{\text{phase + magnitude}} \;\longrightarrow\; \underbrace{S_x(f)}_{\text{magnitude only}}
\;\longrightarrow\; \underbrace{\sigma}_{\text{one number}}$$

Each arrow discards information. The first is why you cannot recover *the* original waveform from a PSD — only a
statistically equivalent one. The second is why equal $\sigma$ says almost nothing about how two noise sources
behave.